# Initialization

## Loading Data

In [6]:
with open(r"/home/chenzhb/Workspaces/verl/mcts_utils/data/qwen2.5-3b_mcts_responses.jsonl", 'r', encoding='utf-8') as f:
    dataset = f.readlines()

## Model Initialization

In [1]:
from openai import OpenAI
import requests

class LLM:
    def __init__(self, base_url="http://localhost:4869/v1", api_key="EMPTY"):
        self.base_url = base_url
        self.api_key = api_key
        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
        )
        self.health_check()

    def generate(self, data, model='qwen2.5-3b', args=None):
        chat_response = self.client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": data},
            ],
            max_tokens=args['max_tokens'],
            temperature=args['temperature'],
            top_p=args['top_p'],
        )
        return chat_response.choices[0].message.content
    
    def health_check(self):
        health_url = self.base_url.replace("/v1", "") + "/health"
        response = requests.get(health_url)
        if response.status_code == 200:
            print("LLM API is available.")
        else:
            print(f"LLM API health check failed with status code: {response.status_code}")

In [2]:
llm = LLM()

LLM API is available.


In [3]:
args = {
    "max_tokens": 4096,
    "temperature": 0.1,
    "top_p": 0.8,
}

## Tool Function

In [ ]:
import subprocess
import sys

def run_code_in_subprocess(code_string):
    # print("CAN: 正在子进程中启动代码...")
    
    try:
        # 使用当前 python 解释器 (sys.executable) 执行代码
        # capture_output=True 会同时捕获 stdout 和 stderr
        # text=True 会让结果以字符串形式返回，而不是字节
        result = subprocess.run(
            [sys.executable, "-c", code_string],
            capture_output=True,
            text=True,
            timeout=10 # 设置超时防止死循环
        )
        
        if result.returncode == 0:
            return {
                "success": True,
                "output": result.stdout,
                "error": None
            }
        else:
            # 如果 returncode != 0，说明出错了
            # stderr 通常包含 Traceback
            return {
                "success": False,
                "output": result.stdout,
                "error": result.stderr 
            }
            
    except subprocess.TimeoutExpired:
        return {"success": False, "error": "CAN: 代码执行超时！"}
    except Exception as e:
        return {"success": False, "error": str(e)}

# 测试代码
dangerous_code = """
import sys
def foo():
    bar()
def bar():
    raise ValueError("Deep failure inside subprocess!")
foo()
"""

res = run_code_in_subprocess(dangerous_code)
if not res["success"]:
    # print("\nCAN: 子进程报错捕获成功：")
    print(res["error"])


In [ ]:
expression = ""
code = wrap_z3_code(declaration,expression)
out = execute_and_capture_output(code)
out


def correct_z3_code(code, error):
    fix_bug_prompt = """
    You are an experienced Python programmer who excels at fixing bugs in code. Given a piece of Python code regarding the use of Z3 solver to verify logical correctness and its error message, please comprehensively judge what went wrong with this code, and place the fixed, syntactically correct, and executable Python code into a ```python ``` block
    ### Given Python Code
    ${code}
    
    ### Error Message
    ${error}
    
    """
    from string import Template
    fix_T = Template(fix_bug_prompt)
    fix_prompt = fix_T.safe_substitute(code=code, error=error)
    fix_out = llm.generate(data=fix_prompt, args=args)
    # print(fix_out)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, fix_out, re.DOTALL)
    return matches[-1]

def correct_loop(code,res):
    code = wrap_z3_code(declaration,expression)
    res = run_code_in_subprocess(code)
    while not res["success"]:
        code = correct_z3_code(code,res["error"])
        # print(new_code)
        res = run_code_in_subprocess(code)
        print(res)

correct_loop(code,out)

# Step 1 : Coreference Resolution

In [ ]:
coref_template = """
You are an expert in the field of Natural Language Processing. Given a piece of text, your task is to perform coreference resolution on the text, replacing all pronouns in the text with specific entities.
Here are an example:


## Input Text

Alice and Bob went to the beach. They were excited to see the waves and the sun. After some time, Alice said to Bob, "Let's build a sandcastle!" He agreed and started gathering some sand. While they were working, a seagull swooped down and tried to steal their snacks.


## Output

Alice and Bob went to the beach. Alice and Bob were excited to see the waves and the sun. After some time, Alice said to Bob, "Let's build a sandcastle!" Bob agreed and started gathering some sand. While Alice and Bob were working, a seagull swooped down and tried to steal Alice and Bob's snacks.
---

Now, please perform Coreference Resolution on the following paragraph:


## Input Text
${text}

## Output


"""
from string import Template
t = Template(coref_template)

In [ ]:
coref_prompt = t.safe_substitute(text=text)
coref_response = llm.generate(data=coref_prompt, args=args)
pprint(coref_response)

# Step 2: Declaration Extraction

In [ ]:
declaration_prompt = """
To accurately translate the context into a first-order logical expression, please analyze and extract the Declaration from the context. Please refer to the following example:

---

## Input
Four boys—Fred, Juan, Marc, and Paul—and three girls—Nita, Rachel, and Trisha—will be assigned to a row of five adjacent lockers, numbered consecutively 1 through 5, arranged along a straight wall. The following conditions govern the assignment of lockers to the seven children: Each locker must be assigned to either one or two children, and each child must be assigned to exactly one locker. Each shared locker must be assigned to one girl and one boy. Juan must share a locker, but Rachel cannot share a locker. Nita's locker cannot be adjacent to Trisha's locker. Fred must be assigned to locker 3.",

## Output
children = EnumSort([Fred, Juan, Marc, Paul, Nita, Rachel, Trisha])
lockers = EnumSort([1, 2, 3, 4, 5])
assigned = Function([children] -> [lockers])

---
## Input
A music store carries exactly ten types of CDs—both new and used of each of jazz, opera, pop, rap, and soul. The store is having a sale on some of these types of CDs. The following conditions must apply: Used pop is on sale; new opera is not. If both types of pop are on sale, then all soul is. If both types of jazz are on sale, then no rap is. If neither type of jazz is on sale, then new pop is. If either type of rap is on sale, then no soul is.

## Output
types = EnumSort([new_jazz, used_jazz, new_opera, used_opera, new_pop, used_pop, new_rap, used_rap, new_soul, used_soul])
on_sale = Function([types] -> [bool])

---
Now, please extract the Declaration from the following problem  and put the Declaration into ```python``` block :

## Input
${context}

"""

In [ ]:
def declaration_ext(declaration_prompt, text):
    from string import Template
    declaration_T = Template(declaration_prompt)
    declaration_prompt = declaration_T.safe_substitute(context = text)
    declarations = llm.generate(data=declaration_prompt,args=args)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, declarations, re.DOTALL)
    return matches[-1]